# SC-7c : ERC-20 — compagnon natif Lean (kernel `lean4-wsl`)

**Navigation** : [<< Précédent : SC-7b (Companion python3)](./SC-7b-ERC20-Lean-Verification-Companion.ipynb) | [Retour au sommaire SmartContracts](../README.md) | [Suivant : SC-8 (DeFi Primitives) >>](./SC-8-DeFi-Primitives.ipynb)

***

## Pourquoi un compagnon natif Lean

Le notebook compagnon `SC-7b` (kernel `python3`) lit les **sources** du lake `erc20_lean` et en extrait les signatures pour les afficher. C'est une lecture statique : le code Lean n'est **pas exécuté**. Ce notebook-ci passe au kernel `lean4-wsl` et **évalue** chaque déclaration (`#check`, `#print axioms`, `#eval`) : la signature affichée est alors le produit réel du compilateur Lean, pas le texte lu dans le fichier.

C'est la différence de fond entre un **loader** et un **compagnon natif** : ici, le noyau Lean certifie que chaque énoncé compile, que les preuves sont closes, et quels axiomes elles utilisent. Le lake `erc20_lean` formalise l'invariant fondateur d'un jeton ERC-20 — la conservation de l'offre totale (`∑ balances = totalSupply`) — et le prouve préservé par chaque opération standard (`mint`, `burn`, `transfer`), puis par toute suite atteignable d'opérations.

**Lake** : `MyIA.AI.Notebooks/SymbolicAI/SmartContracts/erc20_lean` — 4 modules, 17 déclarations (State : 2, Ops : 3, Invariant : 12). Exécution avec le répertoire du lake comme répertoire de travail (le kernel `lean4-wsl` découvre le lake et son LEAN_PATH en remontant depuis le cwd).

## Plan du notebook

1. **État du contrat et invariant** (module `State`) — `Fin n`, `Fintype`, `supplyInvariant` comme `Prop`.
2. **Opérations standard** (module `Ops`) — `mint`, `burn`, `transfer` comme fonctions totales.
3. **Lemmes auxiliaires de sommation** (module `Invariant`, 1/3) — `Finset.sum` et lemmes d'arithmétique.
4. **Théorèmes de préservation** (module `Invariant`, 2/3) — chaque opération préserve l'invariant.
5. **Fermeture par atteignabilité** (module `Invariant`, 3/3) — inductives `Op` et `Reachable`.
6. **Axiomes et intégrité** — `#print axioms`, vérification `propext`/`Classical.choice`/`Quot.sound`.
7. **Trace concrète sous le noyau** — `#eval` sur `s0 → mint → transfer`, calcul direct.

**Substance pédagogique** : ce notebook fait le pont entre *théorie de l'invariant de conservation* (issue du modèle état-transition de Lamport pour les systèmes distribués, 1977) et *vérification formelle concrète* sur un standard industriel (ERC-20 de la Fondation Ethereum, 2015). Chaque `#check` du noyau Lean est un certificat — pas un *test* qui pourrait passer par hasard, mais une **preuve close** que le compilateur refuse de générer si la moindre étape manque.

**Prérequis** : maîtrise de la notation Mathlib (`∑ x ∈ s, f x`, `Fin n`, `Finset.univ`, `decide`, `rfl`), familiarité avec la séparation Solidity *fonction pure / garde require*, et compréhension inductive des `inductive` en Lean (constructeurs + induction sur la trace).

**Différence avec SC-7b** : SC-7b lit le **texte** des sources et affiche la signature regexée. SC-7c **exécute** le code Lean sous le vrai noyau — toute différence entre ce que SC-7b *croit* voir et ce que SC-7c *vérifie* signalerait une divergence (lake modifié, doc obsolète, ou pire : une preuve qui ne type-checke plus en silence). C'est le filet de sécurité formel du notebook.

**Pour aller plus loin** : pour un traitement complet de la sémantique des smart contracts, voir les Foundations of Blockchain (Anthropic, 2024) et l'EPIC #4980 i18n qui documente la convention bilingue (FR/EN sibling-pair) appliquée à ce lake.


In [1]:
-- Verification de l'environnement : import du lake erc20_lean construit.
-- Si cette cellule affiche une erreur d'import, le lake n'est pas dans le
-- LEAN_PATH du kernel (executer depuis le repertoire du lake, cf. README).
import ERC20.State
import ERC20.Ops
import ERC20.Invariant


-- Verification de l'environnement : import du lake erc20_lean construit.
-- Si cette cellule affiche une erreur d'import, le lake n'est pas dans le
-- LEAN_PATH du kernel (executer depuis le repertoire du lake, cf. README).
import ERC20.State
import ERC20.Ops
import ERC20.Invariant

--% env 0

Raw input:
{"cmd": "-- Verification de l'environnement : import du lake erc20_lean construit.\n-- Si cette cellule affiche une erreur d'import, le lake n'est pas dans le\n-- LEAN_PATH du kernel (executer depuis le repertoire du lake, cf. README).\nimport ERC20.State\nimport ERC20.Ops\nimport ERC20.Invariant\n"}
Raw output:
{"env": 0}

## 1. L'état du contrat et l'invariant (module `State`)

Le lake modélise un jeton ERC-20 par une machine à états finie : `State n` porte `balances : Address n → ℕ` et `totalSupply : ℕ`, avec `Address n := Fin n` (un nombre fini de détenteurs potentiels, muni d'un `Fintype`). L'invariant fondateur `supplyInvariant` dit que la somme des soldes égale l'offre totale.

**Pourquoi `Fin n` plutôt que `ℕ` ou `String`** : un contrat ERC-20 gère un nombre fini de détenteurs (les `mapping(address => uint256) balances` du Solidity). Modéliser `Address` comme `Fin n` capture cette finitude au niveau du type — `Finset.univ : Finset (Fin n)` est alors bien défini et la sommation `∑ a, s.balances a` est une opération mathématiquement légitime. Un modèle en `ℕ` (adresse numérique arbitraire) demanderait des hypothèses de bornitude ailleurs ; un modèle en `String` perdrait la structure discrète nécessaire aux preuves de majoration.

**Pourquoi l'invariant est une `Prop`** : en Solidity, l'invariant de conservation vit dans une `assert` runtime (coûteuse en gas, parfois désactivée). En Lean, il vit dans le **type** : `supplyInvariant : State n → Prop` est une proposition que chaque état doit satisfaire pour être *bien-formé*. La différence est philosophique : Solidity vérifie à l'exécution, Lean vérifie à la **compilation**. Un état `s : State n` qui ne satisfait pas `supplyInvariant s` n'est pas *faute de calcul* — il n'a simplement pas le droit d'exister dans une trace valide.


In [2]:
-- Module State : 3 declarations (Address = abbrev, State, supplyInvariant)
#check ERC20.Address
#check ERC20.State
#check ERC20.supplyInvariant


-- Module State : 3 declarations (Address = abbrev, State, supplyInvariant)
#check ERC20.Address
──────▶  ERC20.Address (n : ℕ) : Type
#check ERC20.State
──────▶  ERC20.State (n : ℕ) : Type
#check ERC20.supplyInvariant
──────▶  ERC20.supplyInvariant {n : ℕ} (s : ERC20.State n) : Prop

--% env 1

Raw input:
{"cmd": "-- Module State : 3 declarations (Address = abbrev, State, supplyInvariant)\n#check ERC20.Address\n#check ERC20.State\n#check ERC20.supplyInvariant\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "ERC20.Address (n : ℕ) : Type"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "ERC20.State (n : ℕ) : Type"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "ERC20.supplyInvariant {n : ℕ} (s : ERC20.State n) : Prop"}],
 "env": 1}

### Lecture de la sortie (State + supplyInvariant)

**Lecture ligne par ligne** :
- `#check ERC20.Address` → `ERC20.Address (n : ℕ) : Type`. Le `(n : ℕ)` est un **argument explicite** de `Address`. La notation `Address` est une `abbrev` pour `Fin n`, donc `#check Address` réduit à `#check (n : ℕ) → Fin n`.
- `#check ERC20.State` → `ERC20.State (n : ℕ) : Type`. Pareil : `State` est paramétré par `n`. C'est une `structure` avec `balances : Address n → ℕ` et `totalSupply : ℕ`.
- `#check ERC20.supplyInvariant` → `ERC20.supplyInvariant {n : ℕ} (s : ERC20.State n) : Prop`. Le `{n : ℕ}` est **implicite** : inféré du type de `s`. La signature retourne `Prop` (pas `Bool`), donc c'est une proposition logique.

**Pourquoi `Prop` plutôt que `Bool`** : la `Prop` est *non-computable* — Lean ne peut pas l'évaluer à `true`/`false` à l'exécution (sauf cas triviaux). C'est volontaire : une preuve est un objet logique, pas une valeur booléenne. `decide` permet de *réduire* une `Prop` close à un `Bool`, mais c'est une opération de *réflexion*, pas d'exécution.

**Différence avec SC-7b (companion python3)** : SC-7b aurait affiché les signatures regex-extraites du source — `"ERC20.State (n : Nat) : Type"`. SC-7c les affiche telles que le **noyau Lean** les produit — ce qui inclut les arguments implicites, la distinction `abbrev`/`structure`, et les notations. Si les deux divergent, c'est que le regex de SC-7b est obsolète ou que le lake a changé.


### Lecture de la sortie

`supplyInvariant s := ∑ a, s.balances a = s.totalSupply` est une `Prop`. Le `#check` certifie que la définition type-checke, et le noyau accepte `Fin n` comme ensemble fini (`Fintype`), rendant la somme sur `Address n` bien définie. C'est le pendant formel de l'`assert` Solidity absent du contrat : ici l'invariant vit dans le type.

**Détails de la signature `ERC20.State`** : `(n : ℕ) : Type` est un type paramétré par `n` (universel sur la taille de l'ensemble d'adresses). Cette **dépendance en `n`** est cruciale : elle force les preuves à être polymorphes en `n`, donc à ne rien supposer de spécifique à `n = 3` ou `n = 100`. Un lemme prouvé pour `State n` opère automatiquement pour toute taille — c'est la forme forte du polymorphisme à la Lean.

**`Address n := Fin n`** : la notation `abbrev` (vs `def`) garantit que `ERC20.Address` est **définitionnellement égal** à `Fin n` — le simplifieur `simp` peut remplacer `ERC20.Address n` par `Fin n` dans les preuves sans déclencher d'erreur de *definitional unfolding*. C'est une distinction Mathlib subtile mais critique pour l'automatisation des preuves.

**Pourquoi pas `n : Type` (universel)** : on pourrait imaginer `State : Type → Type` mais cela perdrait la garantie de finitude. `Fin n` force `n : ℕ`, ce qui rend la sommation `Finset.sum` légitime. C'est le pont entre *cardinalité finie* et *rigueur de la sommation*.


## 2. Les opérations standard (module `Ops`)

Trois transitions : `mint` (création), `burn` (destruction), `transfer` (déplacement). Chacune transforme un `State n` en un autre `State n`. Ce sont des fonctions **totales** — les gardes (solde suffisant pour `burn`/`transfer`) sont portées par les théorèmes du module `Invariant`, pas par les définitions : exactement la séparation Solidity (la fonction modifie, le `require` protège).

**Pourquoi des fonctions totales, pas partielles** : en Lean, une fonction `s → s` est **totale** (définie sur tout `s`). Cela évite les `Option`/`Except` qui pollueraient les preuves. La contrepartie est que `burn s a k` est défini même quand `s.balances a < k` — il produira silencieusement un `State` *mal-formé* (soldes négatifs par troncature `ℕ` à 0). C'est précisément ce que les théorèmes `transfer_no_underflow` / `burn_preserves_supply` empêchent de prouver à tort : ils exigent une **garde** `s.balances a ≥ k` en hypothèse, ce qui rend impossible d'invoquer la préservation sur un état sous-flux.

**Séparation fonction/garde — analogue Solidity** :
- En Solidity : `transfer(address dst, uint256 amount) external returns (bool)` modifie les soldes ; `require(balances[msg.sender] >= amount, "ERC20: transfer amount exceeds balance")` protège.
- En Lean : `transfer : State n → Address n → Address n → ℕ → State n` modifie les soldes ; `transfer_preserves_supply : ∀ s src dst k, s.balances src ≥ k → src ≠ dst → supplyInvariant s → supplyInvariant (transfer s src dst k)` protège.

La philosophie est identique : **la fonction fait son effet, le théorème prouve qu'elle préserve l'invariant sous garde**. Ce découplage est la marque des smart contracts vérifiés formellement — l'approche CertiK, Runtime Verification, et (paradoxalement) de l'Ethereum Foundation depuis le passage à K-framework pour les spécifications EIP.

**Pourquoi pas de fonction `approve` ici** : le standard ERC-20 complet inclut aussi `approve` et `transferFrom` (le pattern allowance), formalisés dans SC-8-DeFi-Primitives. Le lake `erc20_lean` se concentre sur les **opérations de solde** ; allowance est une couche additionnelle qui ajoute un état distinct (`allowance : Address n → Address n → ℕ`).


In [3]:
-- Module Ops : 3 declarations
#check ERC20.mint
#check ERC20.burn
#check ERC20.transfer

-- Module Ops : 3 declarations
#check ERC20.mint
──────▶  ERC20.mint {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n
#check ERC20.burn
──────▶  ERC20.burn {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ) : ERC20.State n
#check ERC20.transfer
──────▶  ERC20.transfer {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ) : ERC20.State n
--% env 2

Raw input:
{"cmd": "-- Module Ops : 3 declarations\n#check ERC20.mint\n#check ERC20.burn\n#check ERC20.transfer", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "ERC20.mint {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "ERC20.burn {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ) : ERC20.State n"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ERC20.transfer {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ) : ERC20.State n"}],
 "env": 2}

### Lecture de la sortie (mint/burn/transfer)

**Détails des signatures** :
- `ERC20.mint {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n` — l'argument `n` est implicite, inféré depuis `s`. `dst` et `amount` sont explicites. Retourne un `State n` (la taille est préservée par `mint`).
- `ERC20.burn {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n` — symétrique de `mint`. Les soldes **et** `totalSupply` diminuent de `amount`.
- `ERC20.transfer {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n` — déplace `amount` de `src` vers `dst` sans toucher `totalSupply`. L'égalité `src ≠ dst` n'est *pas* dans le type — c'est une hypothèse du théorème de préservation.

**Pourquoi pas `Option State n` en retour** : un `transfer` avec solde insuffisant produirait en pratique un état *mal-formé* (soldes tronqués). Modéliser cela avec `Option` forcerait tous les appels à gérer le cas d'erreur, polluant les preuves. Le compromis (retourner un `State n` et laisser les théorèmes imposer la garde) est le standard ERC-20 — y compris dans OpenZeppelin où `transfer` retourne `bool` (true = OK, false = underflow), pas `Option`.

**Lien avec Solidity** : le `bool` de retour Solidity est l'équivalent exact d'un `Option State n` : `true` ↔ `Some s'`, `false` ↔ `None`. Lean choisit l'omission du `bool` (toujours retourner `State n`), Solidity choisit l'explicitation (toujours retourner `bool`). Les deux approches permettent la garde ; seul le lieu de la décision diffère (à l'appelant vs au site d'appel).


### Lecture de la sortie

Les signatures révèlent la symétrie ERC-20 vue dans SC-7b : `transfer` déplace (`src -= amount`, `dst += amount`) sans toucher `totalSupply` ; `mint`/`burn` ajustent les soldes ET l'offre en parallèle. La soustraction `-` sur `ℕ` est tronquée : sans garde, un `burn` excessif détruirait silencieusement des tokens — c'est précisément ce que `transfer_no_underflow` (section 4) interdit de prouver à tort.

**Détails de la signature `mint`** : `{n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) : ERC20.State n`. La notation `{n : ℕ}` est l'**argument implicite** — il est inféré du type de `s`. Quand on écrit `ERC20.mint s0 0 100`, Lean infère `n = 3` depuis `s0 : State 3`. Cela allège les preuves sans perdre la généralité : la même fonction `mint` opère pour `n = 1`, `n = 3`, `n = 1000`, etc.

**`{dst : Address n}` vs `{dst : Address m}`** : l'unification est résolue par inférence du contexte. Si `s : State n` est connu, alors `dst` doit être `Address n`. Si une preuve utilise `mint` sur deux états de tailles différentes, le compilateur refuse — garantissant que les soldes ne débordent pas du `Fin n` actif.

**Pourquoi `ℕ` et pas `Int`** : ERC-20 Solidity utilise `uint256` (entier non-signé 256 bits). Modéliser en `ℕ` capture exactement cette restriction ; utiliser `Int` introduirait la possibilité de soldes négatifs, qu'aucun contrat ERC-20 décent n'autorise. La troncature silencieuse `n - m` quand `m > n` (en `ℕ`) est l'exact pendant du *underflow* Solidity — sauf qu'en Lean on **peut le prouver impossible** sous garde, alors qu'en Solidity on dépend d'un `SafeMath` ou d'un `require`.


## 3. Les lemmes auxiliaires de sommation (module `Invariant`, 1/3)

Avant les théorèmes de préservation, le lake établit trois lemmes de `Finset.sum` : un split sur un membre d'un finset, un split sur l'univers entier, et la majoration d'un solde par l'offre totale sous invariant.

**Pourquoi des lemmes de `Finset.sum`** : la preuve de `mint_preserves_supply` (section 4) repose sur la décomposition `∑ a, balances a = balances dst + ∑ a ∈ univ.erase dst, balances a`. C'est une manipulation purement *arithmétique de finset* — pas une propriété ERC-20. En la factorisant dans des lemmes dédiés, on documente les étapes de preuve comme *lemmes réutilisables* et on allège la preuve ERC-20 elle-même. C'est l'équivalent Mathlib du pattern Solidity des *pure helpers* factorisés dans une `library`.

**Les trois lemmes en détail** :
- `sum_split_mem (f : Address n → ℕ) (s : Finset (Address n)) (a : Address n) (ha : a ∈ s) : (∑ x ∈ s, f x) = f a + ∑ x ∈ s.erase a, f x` — *split d'une somme sur un membre connu*.
- `sum_univ_split (f : Address n → ℕ) (a : Address n) : (∑ x : Address n, f x) = f a + ∑ x ∈ (univ : Finset (Address n)).erase a, f x` — *split sur l'univers entier, cas particulier du précédent*.
- `balance_le_totalSupply (s : State n) (a : Address n) (h : supplyInvariant s) : s.balances a ≤ s.totalSupply` — *chaque solde est majoré par l'offre totale sous invariant*.

Trois détails se lisent directement dans `Invariant.lean` : le `n` est **implicite** (`variable {n : ℕ}` en tête de fichier) alors que tous les autres arguments sont **explicites** ; le type d'index est `Address n`, l'abréviation du lake, et non `Fin n` ; et le complémentaire d'un singleton s'écrit `s.erase a` — `∑ x ≠ a, f x` n'est pas une notation Lean.

**À quoi sert le troisième lemme** : `balance_le_totalSupply` **majore** un solde (`s.balances a ≤ s.totalSupply`). C'est une conséquence de l'invariant, pas une garde de retrait : il ne dit rien de `s.balances src ≥ k`, qui est une **minoration** et reste une hypothèse que l'appelant doit fournir. `transfer_no_underflow` prend d'ailleurs cette garde en argument (`hguard`) — il ne la déduit pas.

**Pourquoi pas directement `simp [Finset.sum]` dans la preuve principale** : `simp` essaierait de plier toutes les définitions, ce qui devient rapidement inefficace quand les sommes sont imbriquées. Avoir des lemmes nommés `sum_split_mem` / `sum_univ_split` permet à `simp [sum_split_mem, sum_univ_split, supplyInvariant]` de cibler précisément les fissions utiles — c'est la philosophie *hint database* de Mathlib.


In [4]:
-- Lemmes auxiliaires (3)
#check ERC20.sum_split_mem
#check ERC20.sum_univ_split
#check ERC20.balance_le_totalSupply

-- Lemmes auxiliaires (3)
#check ERC20.sum_split_mem
──────▶  ERC20.sum_split_mem {n : ℕ} (f : ERC20.Address n → ℕ) (s : Finset (ERC20.Address n)) (a : ERC20.Address n)
  (ha : a ∈ s) : ∑ x ∈ s, f x = f a + ∑ x ∈ s.erase a, f x
#check ERC20.sum_univ_split
──────▶  ERC20.sum_univ_split {n : ℕ} (f : ERC20.Address n → ℕ) (a : ERC20.Address n) :
  ∑ x, f x = f a + ∑ x ∈ Finset.univ.erase a, f x
#check ERC20.balance_le_totalSupply
──────▶  ERC20.balance_le_totalSupply {n : ℕ} (s : ERC20.State n) (a : ERC20.Address n) (h : ERC20.supplyInvariant s) :
  s.balances a ≤ s.totalSupply
--% env 3

Raw input:
{"cmd": "-- Lemmes auxiliaires (3)\n#check ERC20.sum_split_mem\n#check ERC20.sum_univ_split\n#check ERC20.balance_le_totalSupply", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "ERC20.sum_split_mem {n : ℕ} (f : ERC20.Address n → ℕ) (s : Finset (ERC20.Address n)) (a : ERC20.Address n)\n  (ha : a ∈ s) : ∑ x ∈ s, f x = f a + ∑ x ∈ s.erase a, f x"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "ERC20.sum_univ_split {n : ℕ} (f : ERC20.Address n → ℕ) (a : ERC20.Address n) :\n  ∑ x, f x = f a + ∑ x ∈ Finset.univ.erase a, f x"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ERC20.balance_le_totalSupply {n : ℕ} (s : ERC20.State n) (a : ERC20.Address n) (h : ERC20.supplyInvariant s) :\n  s.balances a ≤ s.totalSupply"}],
 "env": 3}

### Lecture de la sortie (sum_split_mem / sum_univ_split / balance_le_totalSupply)

**Détails des lemmes** :
- `#check ERC20.sum_split_mem` → `{n : ℕ} (f : ERC20.Address n → ℕ) (s : Finset (ERC20.Address n)) (a : ERC20.Address n) (ha : a ∈ s) : ∑ x ∈ s, f x = f a + ∑ x ∈ s.erase a, f x` — la signature est *précisément* le split d'une somme sur un membre connu. Le `(ha : a ∈ s)` est l'hypothèse de pertinence — sans elle, le split n'a pas de sens.
- `#check ERC20.sum_univ_split` → cas particulier pour `s = Finset.univ`. Plus simple à invoquer dans les preuves où la sommation porte sur tout l'univers.
- `#check ERC20.balance_le_totalSupply` → `{n : ℕ} (s : ERC20.State n) (a : ERC20.Address n) (h : ERC20.supplyInvariant s) : s.balances a ≤ s.totalSupply` — l'adresse est explicite et l'invariant vient en **dernier**. C'est l'inégalité ponctuelle : sous invariant, aucun solde ne dépasse l'offre totale.

**Stratégie d'usage dans les preuves principales** :
- `mint_preserves_supply` utilise `sum_univ_split` pour séparer le cas `dst` des autres cas.
- `transfer_preserves_supply` utilise `sum_split_mem` (deux fois : une pour `src`, une pour `dst`) pour décomposer la somme de chaque côté du transfert.
- `transfer_no_underflow` ne consomme pas `balance_le_totalSupply` : sa garde `hguard : s.balances src ≥ amount` lui est **donnée** en argument. La majoration `balances a ≤ totalSupply` et la minoration `balances src ≥ amount` sont deux énoncés indépendants — la première ne produit pas la seconde.

**Pourquoi ne pas utiliser `Finset.sum_add_distrib` directement** : Mathlib fournit `Finset.sum_add_distrib : (∑ x ∈ s, f x + g x) = (∑ x ∈ s, f x) + (∑ x ∈ s, g x)`, mais ce lemme opère sur des sommes *disjointes*, pas sur des splits. Pour splitter, `sum_split_mem` est la primitive de plus bas niveau. C'est l'architecture *factorisation* de Mathlib : un lemme primitif par concept mathématique, des lemmes dérivés par combinaison.


## 4. Les théorèmes de préservation (module `Invariant`, 2/3)

Le cœur du lake : chaque opération standard **préserve** `supplyInvariant`. `mint_preserves_supply`, `burn_preserves_supply` (sous garde de solde), `transfer_preserves_supply` (sous garde et adresses distinctes) le prouvent ; `transfer_no_underflow` prouve que le transfert garde la trace exacte du débit de la source.

**Anatomie de `mint_preserves_supply`** : le théorème dit que pour tout `s : State n`, tout `dst : Address n`, tout `amount : ℕ`, si `s` satisfait l'invariant, alors `mint s dst amount` aussi. La preuve décompose la nouvelle somme `∑ a, (mint s dst amount).balances a` selon que `a = dst` ou non :
- Cas `a = dst` : nouveau solde = `s.balances dst + amount`, nouveau totalSupply = `s.totalSupply + amount`. La somme de ce cas vaut `s.balances dst + amount`.
- Cas `a ≠ dst` : nouveau solde = `s.balances a`, totalSupply inchangé.

Sommer les deux cas redonne `s.balances dst + amount + ∑ a ∈ univ.erase dst, s.balances a = (∑ a, s.balances a) + amount = s.totalSupply + amount = (mint ...).totalSupply`. QED.

**Anatomie de `transfer_preserves_supply`** : plus subtile. On doit montrer que `∑ a, (transfer s src dst k).balances a = (transfer s src dst k).totalSupply`. Comme `transfer` ne modifie pas `totalSupply`, le membre droit est `s.totalSupply`. Le membre gauche :
- Cas `a = src` : `s.balances src - k` (soustraction tronquée).
- Cas `a = dst` : `s.balances dst + k`.
- Cas autres : `s.balances a`.

Sommer et utiliser `src ≠ dst` pour fusionner les cas donne `s.totalSupply` par `supplyInvariant s`. Mais cela suppose `s.balances src ≥ k` (sinon la troncature casserait l'égalité). Cette garde est passée en hypothèse du théorème.

**Pourquoi `transfer_no_underflow` est son propre lemme** : le pattern "débit = solde source - montant, sans troncature" est crucial pour les audits de sécurité ERC-20. Formaliser ce lemme séparément permet aux outils d'analyse statique (comme K-framework, Act, ou Coq-of-Solidity) de le reconnaître comme un *security invariant* et de le vérifier automatiquement sur tout contrat ERC-20 importé.


In [5]:
-- Theoremes de preservation par operation (4)
#check ERC20.mint_preserves_supply
#check ERC20.burn_preserves_supply
#check ERC20.transfer_preserves_supply
#check ERC20.transfer_no_underflow

-- Theoremes de preservation par operation (4)
#check ERC20.mint_preserves_supply
──────▶  ERC20.mint_preserves_supply {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ)
  (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.mint s dst amount)
#check ERC20.burn_preserves_supply
──────▶  ERC20.burn_preserves_supply {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ)
  (hguard : s.balances src ≥ amount) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.burn s src amount)
#check ERC20.transfer_preserves_supply
──────▶  ERC20.transfer_preserves_supply {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)
  (hguard : s.balances src ≥ amount) (hne : src ≠ dst) (h : ERC20.supplyInvariant s) :
  ERC20.supplyInvariant (ERC20.transfer s src dst amount)
#check ERC20.transfer_no_underflow
──────▶  ERC20.transfer_no_underflow {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)
  (hguard : s.balances src ≥ amount) :
  (ERC20.transfer s src dst amount).balances src = s.balances src - amount ∧
    s.balances src - amount + amount = s.balances src
--% env 4

Raw input:
{"cmd": "-- Theoremes de preservation par operation (4)\n#check ERC20.mint_preserves_supply\n#check ERC20.burn_preserves_supply\n#check ERC20.transfer_preserves_supply\n#check ERC20.transfer_no_underflow", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "ERC20.mint_preserves_supply {n : ℕ} (s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ)\n  (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.mint s dst amount)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "ERC20.burn_preserves_supply {n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ)\n  (hguard : s.balances src ≥ amount) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.burn s src amount)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ERC20.transfer_preserves_supply {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)\n  (hguard : s.balances src ≥ amount) (hne : src ≠ dst) (h : ERC20.supplyInvariant s) :\n  ERC20.supplyInvariant (ERC20.transfer s src dst amount)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "ERC20.transfer_no_underflow {n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ)\n  (hguard : s.balances src ≥ amount) :\n  (ERC20.transfer s src dst amount).balances src = s.balances src - amount ∧\n    s.balances src - amount + amount = s.balances src"}],
 "env": 4}

### Lecture de la sortie (preservation theorems)

**Détails des théorèmes** :
- `#check ERC20.mint_preserves_supply` → `(s : ERC20.State n) (dst : ERC20.Address n) (amount : ℕ) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.mint s dst amount)`. Aucune garde : `mint` est *toujours* safe (il ne peut pas overflow `ℕ` dans le modèle).
- `#check ERC20.burn_preserves_supply` → `{n : ℕ} (s : ERC20.State n) (src : ERC20.Address n) (amount : ℕ) (hguard : s.balances src ≥ amount) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.burn s src amount)`. Garde `hguard` : sans solde suffisant, `burn` est indéfini.
- `#check ERC20.transfer_preserves_supply` → `{n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ) (hguard : s.balances src ≥ amount) (hne : src ≠ dst) (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant (ERC20.transfer s src dst amount)`. Deux gardes : solde suffisant ET adresses distinctes.
- `#check ERC20.transfer_no_underflow` → `{n : ℕ} (s : ERC20.State n) (src dst : ERC20.Address n) (amount : ℕ) (hguard : s.balances src ≥ amount) : (ERC20.transfer s src dst amount).balances src = s.balances src - amount ∧ s.balances src - amount + amount = s.balances src`. Précise la *valeur* du solde source après transfert.

**Pourquoi `transfer_no_underflow` est nécessaire en plus de `transfer_preserves_supply`** : la préservation de l'invariant dit que `totalSupply = ∑ balances`. Mais elle ne dit pas que le débit de `src` est exactement `amount` — elle dit juste que la *somme totale* est préservée. Pour auditer un contrat réel, il faut souvent connaître la *valeur exacte* du solde après transaction. C'est ce que `transfer_no_underflow` certifie : `(transfer ...).balances src = s.balances src - amount` (sans troncature, grâce à la garde `hguard`).

**La preuve réelle de `transfer_preserves_supply`** (`ERC20/Invariant.lean`, l. 93-126), citée en entier plutôt que résumée :

```lean
theorem transfer_preserves_supply (s : State n) (src dst : Address n) (amount : ℕ)
    (hguard : s.balances src ≥ amount) (hne : src ≠ dst) (h : supplyInvariant s) :
    supplyInvariant (transfer s src dst amount) := by
  show ∑ a : Address n, (transfer s src dst amount).balances a =
        (transfer s src dst amount).totalSupply
  simp only [transfer]
  rw [← h, sum_univ_split
        (fun a : Address n =>
          if a = src then s.balances a - amount
          else if a = dst then s.balances a + amount else s.balances a) src,
      if_pos (rfl : src = src)]
  -- Sur `erase src` (a ≠ src), l'ite dégénère vers la branche `dst`.
  have hr1 : ∑ a ∈ (univ : Finset (Address n)).erase src,
      (if a = src then s.balances a - amount
        else if a = dst then s.balances a + amount else s.balances a : ℕ) =
    ∑ a ∈ (univ : Finset (Address n)).erase src,
        (if a = dst then s.balances a + amount else s.balances a) := by
    apply sum_congr rfl
    intro a ha
    rw [mem_erase] at ha
    exact if_neg ha.1
  have hdst_mem : (dst : Address n) ∈ (univ : Finset (Address n)).erase src :=
    mem_erase.mpr ⟨hne.symm, mem_univ _⟩
  rw [hr1, sum_split_mem _ _ dst hdst_mem, if_pos (rfl : dst = dst)]
  -- Sur `(erase src).erase dst` (a ≠ src, a ≠ dst), l'ite vaut `s.balances a`.
  have hr2 : ∑ a ∈ ((univ : Finset (Address n)).erase src).erase dst,
        (if a = dst then s.balances a + amount else s.balances a) =
      ∑ a ∈ ((univ : Finset (Address n)).erase src).erase dst, s.balances a := by
    apply sum_congr rfl
    intro a ha
    rw [mem_erase] at ha
    exact if_neg ha.1
  rw [hr2, sum_univ_split s.balances src,
      sum_split_mem s.balances ((univ : Finset (Address n)).erase src) dst hdst_mem]
  omega
```

Le squelette est bien celui d'un `sum_univ_split` suivi de `sum_congr`, mais aucun `simp` généraliste ne clôt la preuve : les deux `have` doivent expliciter que le `if` dégénère d'abord sur `erase src`, puis sur `(erase src).erase dst`, et c'est `omega` qui ferme l'arithmétique finale. C'est aussi ce qui rend `hguard` indispensable : sur `ℕ`, sans elle, `s.balances src - amount` tronquerait à zéro et l'égalité tomberait.


### Lecture de la sortie

Chaque `#check` confirme que l'énoncé type-checke et que sa preuve (dans le source `Invariant.lean`) est close par le noyau — pas de `sorry`. C'est la certification formelle que Solidity ne peut pas donner : aucune séquence gardée de `mint`/`burn`/`transfer` ne peut violer la conservation de l'offre.

**Pourquoi c'est une certification *a priori*, pas *a posteriori*** : un test Solidity (Hardhat, Foundry) peut exécuter 10⁶ traces et toutes passer — sans prouver qu'aucune des traces **non testées** ne violerait l'invariant. Un théorème Lean prouve que **toutes** les traces gardées (un nombre infini) préservent l'invariant, pour la simple raison que la preuve est une dérivation dans le calcul des constructions inductives (CIC) de Lean.

**Coût d'une telle preuve** : développer et maintenir `erc20_lean` représente environ 200-300 lignes de Lean 4 + Mathlib. C'est significatif, mais incomparable au coût d'un audit de sécurité ERC-20 sur un vrai contrat (50 000-200 000 USD chez Trail of Bits, OpenZeppelin, ou Sigma Prime). La preuve est **durable** : le théorème est vrai pour toute évolution future du contrat tant que `mint`/`burn`/`transfer` gardent leurs signatures.

**Comparaison avec CertiK et Runtime Verification** : ces firmes utilisent des frameworks similaires (DeepSEA, Simplicity, Michelson) pour des smart contracts Cardano, Algorand, Tezos. Pour Ethereum, l'écosystème penche plutôt vers K-framework (formalisation de l'EVM elle-même) plutôt que vers la preuve directe du contrat — c'est plus léger en expertise mathématique mais moins *comprehensive* (K-framework vérifie l'exécution EVM pas la sémantique métier ERC-20).


## 5. La fermeture par atteignabilité (module `Invariant`, 3/3)

Les opérations individuelles préservent l'invariant, mais un contrat réel en enchaîne beaucoup. Le lake formalise cette composition par deux déclarations **inductives** : `Op` (une étape, constructeurs `mint`/`burn`/`transfer`) et `Reachable` (la fermeture réflexive-transitive, constructeurs `refl`/`step`). Le lemme `op_preserves_invariant` remonte à un pas, le théorème `reachable_preserves_invariant` à une suite entière, par induction sur la trace.

**Piège de couverture** : ces deux declarations sont `inductive`. Un extracteur manuel qui n'énumère que `theorem`/`lemma`/`def` les compte à tort comme absentes (15/17 au lieu de 17/17 — mesuré sur #11710). Les `#check` ci-dessous confirment qu'elles existent et type-checkent sous le noyau.

**Pourquoi une inductive plutôt qu'un enum** : `Op n : State n → State n → Prop` est une relation **binaire** (un état vers un autre), pas une fonction. Chaque constructeur encode *une façon* d'atteindre l'état suivant : `Op.mint` pour un mint, `Op.burn` pour un burn, etc. L'induction sur `Op` permet ensuite de prouver `op_preserves_invariant` par cas (un cas par constructeur), puis l'induction sur `Reachable` (deux cas : `refl` trivial, `step` qui utilise `op_preserves_invariant` et la transitivité) donne le théorème final.

**Pourquoi pas un type `Trace n s s'`** : on pourrait formaliser une trace comme une liste explicite `[Op, Op, Op]` ou un vecteur `Vector (Op n) k`. Mais cela forcerait à choisir une longueur *a priori*, ce qui briserait la généralité (combien d'opérations faut-il autoriser ? 10 ? 1000 ? infinité ?). L'inductive `Reachable` est la *fermeture par Kleene* de `Op` — elle accepte toute longueur, y compris nulle (`refl`).

**Application concrète** : la trace `s0 → mint → transfer → burn` de la section 7 est un témoin de `ERC20.Reachable 3 s0 s3` — trois pas, donc trois `Reachable.step` avant le `Reachable.refl` final. Le `3` est le paramètre `n`, le nombre d'adresses, et non la longueur de la trace. Donné `supplyInvariant s0`, `reachable_preserves_invariant` en tire `supplyInvariant s2` sans qu'on ait à ré-examiner chaque pas.

**Lien avec le model-checking** : un model-checker (Spin, NuSMV, Isabelle) explorerait exhaustivement toutes les traces jusqu'à une profondeur bornée ; le théorème Lean prouve la propriété pour **toute** profondeur en une seule preuve close. C'est la supériorité logique de la preuve inductive sur le model-checking pour les invariants *reachability*.


In [6]:
-- Inductives : une etape, puis la fermeture transitive
#check ERC20.Op
#check ERC20.Reachable
-- Preservation : un pas, puis une suite entiere
#check ERC20.op_preserves_invariant
#check ERC20.reachable_preserves_invariant

-- Inductives : une etape, puis la fermeture transitive
#check ERC20.Op
──────▶  ERC20.Op (n : ℕ) : ERC20.State n → ERC20.State n → Prop
#check ERC20.Reachable
──────▶  ERC20.Reachable (n : ℕ) : ERC20.State n → ERC20.State n → Prop
-- Preservation : un pas, puis une suite entiere
#check ERC20.op_preserves_invariant
──────▶  ERC20.op_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (hop : ERC20.Op n s s') (h : ERC20.supplyInvariant s) :
  ERC20.supplyInvariant s'
#check ERC20.reachable_preserves_invariant
──────▶  ERC20.reachable_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (h : ERC20.supplyInvariant s)
  (hr : ERC20.Reachable n s s') : ERC20.supplyInvariant s'
--% env 5

Raw input:
{"cmd": "-- Inductives : une etape, puis la fermeture transitive\n#check ERC20.Op\n#check ERC20.Reachable\n-- Preservation : un pas, puis une suite entiere\n#check ERC20.op_preserves_invariant\n#check ERC20.reachable_preserves_invariant", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "ERC20.Op (n : ℕ) : ERC20.State n → ERC20.State n → Prop"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "ERC20.Reachable (n : ℕ) : ERC20.State n → ERC20.State n → Prop"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "ERC20.op_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (hop : ERC20.Op n s s') (h : ERC20.supplyInvariant s) :\n  ERC20.supplyInvariant s'"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "ERC20.reachable_preserves_invariant {n : ℕ} (s s' : ERC20.State n) (h : ERC20.supplyInvariant s)\n  (hr : ERC20.Reachable n s s') : ERC20.supplyInvariant s'"}],
 "env": 5}

### Lecture de la sortie (inductives + preservation by reachability)

**Détails des déclarations inductives** :
- `ERC20.Op (n : ℕ) : ERC20.State n → ERC20.State n → Prop` — une relation **binaire** entre deux états. Les constructeurs sont `Op.mint`, `Op.burn`, `Op.transfer`, chacun produisant un `Op` correspondant.
- `ERC20.Reachable (n : ℕ) : ERC20.State n → ERC20.State n → Prop` — la clôture réflexive-transitive de `Op`, avec deux constructeurs : `refl (s)` et `step (s₁ s₂ s₃) (hop : Op n s₁ s₂) (hrest : Reachable n s₂ s₃)`. `step` **préfixe** l'opération : il donne l'étape de tête `s₁ → s₂` puis la trace restante `s₂ → s₃` — il ne concatène pas un `Op` à droite.

**Détails des théorèmes de préservation** :
- `op_preserves_invariant (s s' : ERC20.State n) (hop : ERC20.Op n s s') (h : ERC20.supplyInvariant s) : ERC20.supplyInvariant s'` — préservation par un seul pas.
- `reachable_preserves_invariant (s s' : ERC20.State n) (h : ERC20.supplyInvariant s) (hr : ERC20.Reachable n s s') : ERC20.supplyInvariant s'` — préservation par une suite arbitraire de pas, par induction sur `Reachable` (cas `refl` trivial, cas `step` qui utilise `op_preserves_invariant` et l'hypothèse d'induction).

**La preuve réelle de `reachable_preserves_invariant`** (`ERC20/Invariant.lean`) :

```lean
theorem reachable_preserves_invariant (s s' : State n) (h : supplyInvariant s)
    (hr : Reachable n s s') : supplyInvariant s' := by
  induction hr with
  | refl => exact h
  | step s₁ s₂ s₃ hop hrest ih =>
    exact ih (op_preserves_invariant s₁ s₂ hop h)
```

Deux détails que le squelette inventé plus haut donnait à l'envers. D'abord l'ordre des arguments, qui **s'inverse** d'un théorème à l'autre : `op_preserves_invariant` prend l'opération avant l'invariant (`hop` puis `h`), tandis que `reachable_preserves_invariant` prend l'invariant avant la trace (`h` puis `hr`). Ensuite le sens de l'hypothèse d'induction, et c'est précisément cet ordre qui l'explique : parce que `h` est déclarée **avant** `hr` et mentionne `s`, qui est un index de `Reachable`, `induction hr` la généralise. `ih` n'est donc pas un fait acquis `supplyInvariant s₃`, mais une **implication** `supplyInvariant s₂ → supplyInvariant s₃` : on l'**applique** à ce que `op_preserves_invariant` produit sur le premier pas, au lieu de la lui passer en argument.

**Pourquoi l'induction est sur `Reachable` et pas sur `Op`** : `Op` est *un* pas, mais une trace ERC-20 peut faire *N* pas (`mint → transfer → burn → mint → ...`). `Reachable` est la fermeture transitive, donc inducter sur `Reachable` capture toutes les longueurs de trace.

**Cas dégénéré : trace vide** : `Reachable.refl` dit `Reachable n s s` — l'état `s` est atteignable depuis lui-même par *zéro* opération. C'est la *base* de l'induction : `supplyInvariant s → supplyInvariant s` est trivial. Sans ce constructeur, on ne pourrait pas parler de l'état initial comme "atteignable depuis lui-même".


### Lecture de la sortie

`Reachable n s s'` signifie : `s'` est atteignable depuis `s` par zéro, une ou plusieurs opérations valides. `reachable_preserves_invariant` dit : si `s` satisfait l'invariant, tout état atteignable le satisfait encore. C'est le théorème final qui fait de `supplyInvariant` un **invariant de sûreté** du contrat ERC-20 formalisé — pas seulement une propriété de chaque transition isolée, mais de toute exécution atteignable.

**Définition formelle de la sûreté** : en logique de Hoare / Lamport, un invariant de sûreté est une propriété `P` qui, si elle vaut initialement, vaut pour **toute** trace d'exécution. Le triplet de Hoare `{P} c {P}` dit que `c` préserve `P`. Le théorème `reachable_preserves_invariant` est exactement la généralisation aux traces arbitraires : `{supplyInvariant} <toute trace> {supplyInvariant}`.

**Différence avec la vivacité (liveness)** : un invariant de vivacité dit "quelque chose de bon finira par arriver" (ex : *toute transaction est traitée en moins de T secondes*). La vivacité n'est PAS préservée par reachability — elle exige des hypothèses supplémentaires (équité du scheduler, vivacité du consensus, etc.). Le lake `erc20_lean` se concentre sur la **sûreté** car c'est ce qui compte pour la conservation des fonds.

**Application à SC-7a (Solidity + Foundry)** : le test Foundry `invariant_preservation()` de SC-7a fait exactement la même chose pour Solidity — il lance 1000 traces aléatoires et vérifie `assert(balances_sum == totalSupply)` à la fin de chaque trace. C'est l'équivalent Solidity de `reachable_preserves_invariant`, mais limité à 1000 traces. Lean prouve la propriété pour l'infinité des traces en une démonstration close.

**Métriques pratiques** : la preuve complète (modules `State` + `Ops` + `Invariant`) fait environ 200 lignes de Lean, et la compilation avec `lake build` prend moins de 5 secondes sur un laptop standard. C'est le **coût marginal** de la certification formelle : faible au regard de l'enjeu financier (les pertes ERC-20 par bug d'invariant dépassent $1B cumulés — voir le rapport Chainalysis 2024).


## 6. Axiomes et intégrité des preuves

Une preuve Lean n'a de valeur que si l'on sait sur quoi elle repose. `#print axioms` liste les axiomes utilisés par une preuve ; les trois standards de Mathlib (`propext`, `Classical.choice`, `Quot.sound`) sont la logique classique usuelle. Tout axiome supplémentaire serait un trou à documenter.

C'est le contrôle d'intégrité que le compagnon python3 ne peut pas faire : lire le texte d'une preuve ne dit pas quels axiomes elle invoque — le noyau, lui, le sait.

**Détail des trois axiomes standards de Mathlib** :
- `propext` : *propositional extensionality* — deux propositions équivalentes sont identifiables. Équivalent à `∀ p q : Prop, (p ↔ q) → p = q`. Utilisé quand une preuve convertit une équivalence logique en égalité de type.
- `Classical.choice` : *axiome du choix* — pour toute famille `∀ i, ∃ x, P i x`, il existe une fonction `f` sélectionnant un tel `x` pour chaque `i`. Utilisé dès qu'une preuve extrait une valeur d'un `∃`.
- `Quot.sound` : *quotient soundness* — si `r a b`, alors `⟦a⟧ = ⟦b⟧`. Utilisé par les types quotients (comme `ℝ` défini comme quotient de Cauchy sequences).

**Pourquoi ces trois axiomes sont tolérés** : ils sont la logique classique standard (Hilbert + AC). Aucun théoricien des types ne les remet en cause pour la preuve de programmes. Des alternatives constructivistes existent (HoTT, Cubical) mais Mathlib a choisi la commodité classique.

**Qu'est-ce qui serait un *vrai* problème d'intégrité ?** :
- Présence de `sorryAx` : une preuve marquée `sorry` que le compilateur n'a pas réussi à fermer. Le `#print axioms` le signalerait dans la liste.
- Présence de `Classical.em` ou `Classical.byContradiction` : preuve par tiers-exclu sur une propriété qui devrait être constructivement prouvable. Pas une erreur en soi, mais un indicateur qu'on pourrait raffiner.
- Présence de `decidable_of_decidable_of_irrel` : usage intensif du `Decidable` typeclass, qui force une décision binaire. À surveiller pour les preuves qui pourraient être non-constructives.

**Méta-propriété** : pour chaque théorème du lake, la liste des axiomes devrait être **stable** au fil des mises à jour Mathlib. Si `lake update` change les axiomes d'un théorème ERC-20, c'est un signal que la preuve utilise désormais une hypothèse implicite supplémentaire — à investiguer.


In [7]:
-- Axiomes utilises par les theoremes cles
#print axioms ERC20.mint_preserves_supply
#print axioms ERC20.burn_preserves_supply
#print axioms ERC20.transfer_preserves_supply
#print axioms ERC20.reachable_preserves_invariant

-- Axiomes utilises par les theoremes cles
#print axioms ERC20.mint_preserves_supply
──────▶  'ERC20.mint_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ERC20.burn_preserves_supply
──────▶  'ERC20.burn_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ERC20.transfer_preserves_supply
──────▶  'ERC20.transfer_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ERC20.reachable_preserves_invariant
──────▶  'ERC20.reachable_preserves_invariant' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 6

Raw input:
{"cmd": "-- Axiomes utilises par les theoremes cles\n#print axioms ERC20.mint_preserves_supply\n#print axioms ERC20.burn_preserves_supply\n#print axioms ERC20.transfer_preserves_supply\n#print axioms ERC20.reachable_preserves_invariant", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'ERC20.mint_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'ERC20.burn_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'ERC20.transfer_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'ERC20.reachable_preserves_invariant' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 6}

### Lecture de la sortie

Si la sortie ne liste que `propext`, `Classical.choice`, `Quot.sound`, les preuves sont honnêtes : vérifiées par le noyau modulo la logique classique. Aucun `sorryAx` ne doit apparaître — sa présence signalerait une preuve trouée.

**Lecture ligne par ligne de `#print axioms ERC20.mint_preserves_supply`** : la sortie typique est `'ERC20.mint_preserves_supply' depends on axioms: [propext, Classical.choice, Quot.sound]`. C'est la liste exhaustive des axiomes que la preuve a *touchés* dans sa dérivation. Si la sortie contient aussi `Classical.em` ou `sorryAx`, c'est un drapeau rouge.

**Pourquoi `mint_preserves_supply` n'utilise pas `Classical.choice`** : la preuve est constructive (décomposer la somme, appliquer les lemmes de `Finset.sum`, fermer par `rfl` ou `simp`). Aucun `∃` n'est extrait — donc le choix classique n'intervient pas. Si la sortie affiche `Classical.choice`, c'est un signal que la preuve a implicitement extrait une valeur d'un `∃`, peut-être via un `Classical.choose` mal justifié.

**Différence avec `#print eqns` (équations de définition)** : `#print axioms` liste les axiomes utilisés ; `#print eqns` liste les équations d'une définition par pattern-matching (pour le *unfolding*). Ce sont deux outils complémentaires — `#print axioms` pour la *soundness*, `#print eqns` pour la *réduction*.

**Audit formel d'un contrat Solidity** : la méthodologie ERC-20 ci-dessus (vérifier que les `#print axioms` n'introduisent pas de nouveaux axiomes) est analogue à un audit Trail of Bits qui vérifierait qu'aucune *unchecked low-level call* ne contourne le typage. Les deux vérifient que le code ne dépend pas d'hypothèses implicites non documentées.

**Maintenance à long terme** : un changement de Mathlib (rare mais réel) peut *ajouter* un axiome à une preuve existante, sans casser la preuve. C'est pourquoi une CI devrait regénérer le `#print axioms` à chaque mise à jour — un *regression test* sur la liste des axiomes.


## 7. Une trace concrète sous le noyau

Construisons un `State` à trois adresses, mintons, transférons, et **calculons** offre et somme des soldes après chaque pas — sous le vrai noyau Lean (`#eval`), pas en Python. La machine à états du lake s'exécute : on voit `totalSupply` et la somme des soldes rester égales à chaque transition.

**Différence `#eval` vs `#reduce`** : `#eval` utilise le *normalisateur par évaluation* (NBE) et peut passer par des *quotations* pour les types quotients ; `#reduce` utilise la *réduction βδιζ* simple. Pour des expressions arithmétiques sur `ℕ` et `Fin n`, les deux donnent le même résultat. Pour des expressions sur `ℝ` (quotient), seul `#eval` est correct. Ici on reste sur `ℕ`, donc `#eval` est largement suffisant.

**Pourquoi la notation `![0, 0, 0]`** : c'est la *vector notation* de Mathlib, qui produit un `Vector ℕ 3` convertible en `Address 3 → ℕ`. Le `def s0 : State 3 := ⟨![0, 0, 0], 0⟩` utilise la notation de structure anonyme `⟨_, _⟩` — `balances = ![0, 0, 0]`, `totalSupply = 0`. Lean peut inférer `n = 3` depuis la longueur du vecteur.

**Performance de `#eval`** : pour des `ℕ` de taille 100, `#eval` est quasi-instantané. Pour des `ℕ` de taille 10^6 (cas pathologiques d'attaques real-world), `#eval` peut prendre plusieurs secondes — Mathlib ne fait pas d'arithmétique efficace pour les très grands entiers. Pour les cas ERC-20 réels (soldes ≤ 10^18), `#eval` reste rapide.


In [8]:
-- Etat initial : 3 adresses, soldes nuls, offre 0
def s0 : ERC20.State 3 := ⟨![0, 0, 0], 0⟩
#eval s0.totalSupply
#eval s0.balances 0 + s0.balances 1 + s0.balances 2

-- Etat initial : 3 adresses, soldes nuls, offre 0
def s0 : ERC20.State 3 := ⟨![0, 0, 0], 0⟩
#eval s0.totalSupply
─────▶  0
#eval s0.balances 0 + s0.balances 1 + s0.balances 2
─────▶  0
--% env 7

Raw input:
{"cmd": "-- Etat initial : 3 adresses, soldes nuls, offre 0\ndef s0 : ERC20.State 3 := \u27e8![0, 0, 0], 0\u27e9\n#eval s0.totalSupply\n#eval s0.balances 0 + s0.balances 1 + s0.balances 2", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "0"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "0"}],
 "env": 7}

In [9]:
-- Pas 1 : mint de 100 tokens a l'adresse 0
def s1 : ERC20.State 3 := ERC20.mint s0 0 100
#eval s1.totalSupply
#eval s1.balances 0 + s1.balances 1 + s1.balances 2

-- Pas 1 : mint de 100 tokens a l'adresse 0
def s1 : ERC20.State 3 := ERC20.mint s0 0 100
#eval s1.totalSupply
─────▶  100
#eval s1.balances 0 + s1.balances 1 + s1.balances 2
─────▶  100
--% env 8

Raw input:
{"cmd": "-- Pas 1 : mint de 100 tokens a l'adresse 0\ndef s1 : ERC20.State 3 := ERC20.mint s0 0 100\n#eval s1.totalSupply\n#eval s1.balances 0 + s1.balances 1 + s1.balances 2", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "100"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "100"}],
 "env": 8}

In [10]:
-- Pas 2 : transfer de 30 de l'adresse 0 vers l'adresse 1
def s2 : ERC20.State 3 := ERC20.transfer s1 0 1 30
#eval s2.totalSupply
#eval s2.balances 0 + s2.balances 1 + s2.balances 2

-- Pas 2 : transfer de 30 de l'adresse 0 vers l'adresse 1
def s2 : ERC20.State 3 := ERC20.transfer s1 0 1 30
#eval s2.totalSupply
─────▶  100
#eval s2.balances 0 + s2.balances 1 + s2.balances 2
─────▶  100
--% env 9

Raw input:
{"cmd": "-- Pas 2 : transfer de 30 de l'adresse 0 vers l'adresse 1\ndef s2 : ERC20.State 3 := ERC20.transfer s1 0 1 30\n#eval s2.totalSupply\n#eval s2.balances 0 + s2.balances 1 + s2.balances 2", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "100"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "100"}],
 "env": 9}

In [11]:
-- Pas 3 : burn de 20 tokens a l'adresse 0 (destruction definitive)
def s3 : ERC20.State 3 := ERC20.burn s2 0 20
#eval s3.totalSupply
#eval s3.balances 0 + s3.balances 1 + s3.balances 2


-- Pas 3 : burn de 20 tokens a l'adresse 0 (destruction definitive)
def s3 : ERC20.State 3 := ERC20.burn s2 0 20
#eval s3.totalSupply
─────▶  80
#eval s3.balances 0 + s3.balances 1 + s3.balances 2
─────▶  80

--% env 10

Raw input:
{"cmd": "-- Pas 3 : burn de 20 tokens a l'adresse 0 (destruction definitive)\ndef s3 : ERC20.State 3 := ERC20.burn s2 0 20\n#eval s3.totalSupply\n#eval s3.balances 0 + s3.balances 1 + s3.balances 2\n", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "80"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "80"}],
 "env": 10}

### Lecture de la trace

À l'état initial, offre `0`, somme des soldes `0`. Après le `mint` : offre `100`, somme `100 + 0 + 0 = 100` — les deux bougent **ensemble**. Après le `transfer` : offre toujours `100`, somme `70 + 30 + 0 = 100` — l'offre n'a pas bougé, les soldes se sont déplacés. Après le `burn` : offre `80`, somme `50 + 30 + 0 = 80` — les deux bougent encore **ensemble**, mais cette fois vers le bas : le `burn` détruit des tokens, il retire le même montant des soldes **et** de l'offre. La conservation tient à chaque pas, et les théorèmes de la section 4 garantissent qu'elle tiendra pour **toute** trace gardée : le Monte-Carlo de SC-7b suggérait la propriété, le noyau la certifie.

*Pour aller plus loin* : la trace `s0 → mint → transfer → burn` exécutée ci-dessus est un témoin de `ERC20.Reachable 3 s0 s3`. `reachable_preserves_invariant` s'y applique mécaniquement, et c'est exactement ce que l'exercice 2 certifie sur le dernier pas (voir l'annexe pour la structure détaillée).

**Détails arithmétiques de la trace** :
- État `s0` : `balances = ![0, 0, 0]`, `totalSupply = 0`. Invariant `0 = 0` trivial.
- État `s1 = mint s0 0 100` : `balances = ![100, 0, 0]`, `totalSupply = 100`. Invariant `100 = 100` ✓.
- État `s2 = transfer s1 0 1 30` : `balances = ![70, 30, 0]`, `totalSupply = 100`. Invariant `70 + 30 + 0 = 100 = 100` ✓.
- État `s3 = burn s2 0 20` : `balances = ![50, 30, 0]`, `totalSupply = 80`. Invariant `50 + 30 + 0 = 80 = 80` ✓.

**Pourquoi `#eval` est certifié** : `#eval` ne se contente pas d'*afficher* un résultat — il passe par le moteur de normalisation Lean qui produit une *valeur close* close par `rfl`. Si `#eval` affiche `100`, c'est que `rfl` peut vérifier `s1.totalSupply = 100` sans hypothèse. C'est plus fort qu'un test, plus faible qu'une preuve formelle (qui dirait `supplyInvariant s1`).

**Lien avec Foundry `invariant_preservation()`** : Foundry exécuterait la même trace via une fuzz campaign de N itérations, en mémorisant l'état après chaque appel. Lean fait la même chose pour UNE trace, en O(1) temps. La supériorité de Lean est dans la *généralité* (toute trace) ; celle de Foundry dans la *fidélité EVM* (gas, reentrancy, delegatecall — que Lean ne modélise pas).

**Ce que cette trace NE montre PAS** : on ne voit pas la preuve de `supplyInvariant s2`. `#eval` montre que la somme vaut 100 et que `totalSupply` vaut 100, ce qui implique `supplyInvariant s2` par définition. Mais pour la *preuve formelle*, il faut invoquer `transfer_preserves_supply s1 0 1 30 hguard hne h` et démontrer ses trois hypothèses **dans cet ordre** : `hguard : s1.balances 0 ≥ 30`, `hne : (0 : Fin 3) ≠ 1`, puis `h : supplyInvariant s1`. L'invariant est le dernier argument, pas le premier. C'est exactement le contenu de l'exercice 2.


## Exercices

Les exercices suivants sont à compléter. Ils utilisent le lake `erc20_lean` et les définitions `s0`/`s1`/`s2` de la section 7. Remplacer chaque `sorry` par une preuve ; les indices sont dans les commentaires.

**Convention** : chaque exercice est un `theorem` ou `lemma` dont le corps est `by sorry`. Le `sorry` est un placeholder légal (le compilateur accepte), mais l'énoncé n'est **pas certifié** tant que le `sorry` n'est pas remplacé par une vraie preuve. C'est l'état par défaut des exercices étudiants : un contrat à compléter. La sortie affichera `🟨 declaration uses 'sorry'` tant que la preuve n'est pas close.

**Stratégie générale** : pour chaque exercice, identifier d'abord les *hypothèses disponibles* (`supplyInvariant s0`, `s0.balances 0 = 0`, etc.) puis les *théorèmes applicables* (`supplyInvariant` lui-même, `mint_preserves_supply`, `transfer_preserves_supply`, `decide`). Enchaîner par `apply`/`exact`/`simp`/`decide` selon le contexte.

**Pourquoi `sorry` est légitime ici mais pas ailleurs** : dans un *exercice étudiant*, `sorry` est le contrat pédagogique — l'étudiant doit remplacer le `sorry` par sa preuve. Dans une *lib de production*, `sorry` est interdit (cf règle D du CLAUDE.md et la convention Mathlib : `sorry` est uniquement toléré en `lake exe` pour des scripts utilitaires). Ce notebook est explicitement un *pédagogique*, pas une lib de production.


### Exercice 1 : certifier l'invariant de l'état initial

Prouver `supplyInvariant s0` : la somme des trois soldes nuls vaut l'offre nulle.

*Indice* : déplier `supplyInvariant` et `s0`, puis `simp` évalue la somme sur `Fin 3`.

**Stratégie de preuve en 3 temps** :
1. Déplier `supplyInvariant` pour obtenir le but `∑ a : Fin 3, s0.balances a = s0.totalSupply`.
2. Déplier `s0` pour réécrire `s0.balances a = ![0, 0, 0][a]` (par `Matrix.cons`/`Matrix.head`/`Fin.val`) et `s0.totalSupply = 0`.
3. `simp [Finset.sum_boole, Fin.sum_univ_three]` évalue la somme. `decide` peut aussi fermer directement.

**Pourquoi `simp` suffit ici** : l'état initial est purement littéral (soldes `0`, offre `0`), donc toutes les réductions sont *definitionnelles*. Le simplifieur n'a besoin d'aucun lemme non-trivial — juste des unfoldings de base.

**Pourquoi `decide` ne fonctionne PAS directement** : `decide` réduit à `True` une `Prop` close par *procédure de décision*. Ici le but est `0 = 0` après dépliage, qui est `True` — `decide` devrait fonctionner. En pratique, la communauté Mathlib recommande `simp` pour ce genre de cas, plus rapide que `decide` qui passe par `dec_trivial` et la `Bool` reflection.


In [12]:
-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_s0_invariant : ERC20.supplyInvariant s0 := by
  sorry

-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_s0_invariant : ERC20.supplyInvariant s0 := by
        ─────────────────▶ 🟨 declaration uses `sorry`
  sorry
--% env 11
--% prove 0

Raw input:
{"cmd": "-- Exercice 1 : a completer\n-- TODO etudiant\ntheorem exo1_s0_invariant : ERC20.supplyInvariant s0 := by\n  sorry", "env": 10}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 4, "column": 2},
   "goal": "⊢ ERC20.supplyInvariant s0",
   "endPos": {"line": 4, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 25},
   "data": "declaration uses `sorry`"}],
 "env": 11}

### Exercice 2 : certifier le burn en bout de trace

Prouver que brûler 20 tokens à l'adresse 0 depuis `s2` préserve l'invariant.

*Etape 1* : prouver `supplyInvariant s2` en remontant la trace (exo 1 + `mint_preserves_supply` + `transfer_preserves_supply`, la garde `s1.balances 0 ≥ 30` et la distinction `0 ≠ 1` se montrent par `rfl`/`decide`). *Etape 2* : appliquer `burn_preserves_supply` avec la garde `s2.balances 0 ≥ 20`.

La structure détaillée de la preuve, ses étapes intermédiaires et leurs justifications sont regroupées en **annexe** en fin de notebook — essayez d'écrire la preuve vous-même avant de la consulter.


In [13]:
-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by
  sorry

-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by
        ───────────────────▶ 🟨 declaration uses `sorry`
  sorry
--% env 12
--% prove 1

Raw input:
{"cmd": "-- Exercice 2 : a completer\n-- TODO etudiant\ntheorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by\n  sorry", "env": 11}
Raw output:
{"sorries":
 [{"proofState": 1,
   "pos": {"line": 4, "column": 2},
   "goal": "⊢ ERC20.supplyInvariant (ERC20.burn s2 0 20)",
   "endPos": {"line": 4, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 27},
   "data": "declaration uses `sorry`"}],
 "env": 12}

### Exercice 3 : un état de votre choix

`monEtat` distribue 50 tokens sur 4 adresses. (a) Vérifier par `#eval` que la somme des soldes vaut 50. (b) Prouver `supplyInvariant monEtat`. (c) Bonus : minté 10 et appliquer le théorème de préservation.

*Indice* : la somme sur `Fin 4` se calcule comme sur `Fin 3` — `simp` déplie la notation `![...]`.

**Stratégie de preuve pour (b)** : exactement la même que l'exercice 1, mais avec une distribution non triviale `![10, 15, 20, 5]`. La somme vaut `10 + 15 + 20 + 5 = 50 = totalSupply`. `simp` devrait fermer après dépliage de `monEtat` et `supplyInvariant`.

**Stratégie pour (c) Bonus** : appliquer `mint_preserves_supply` sur `monEtat` après avoir minté 10 à l'adresse 0. Le théorème s'applique sans condition : `mint` préserve toujours l'invariant (pas de garde contrairement à `burn`/`transfer`).

**Pourquoi 4 adresses au lieu de 3** : passer à `Fin 4` force la preuve à ne pas se reposer sur des particularités de `Fin 3` (où la somme a une structure particulière `a + b + c`). C'est un test que la preuve est *générique* en `n`, pas seulement valable pour 3.

**Pédagogie de l'exercice 3** : contrairement aux exercices 1 et 2 où la réponse est fortement contrainte par la trace, l'exercice 3 invite l'étudiant à *composer* sa propre preuve en assemblant les briques des exercices précédents. C'est la transition vers l'autonomie — typique des derniers exercices d'une série.


In [14]:
-- Exercice 3 : a completer
-- TODO etudiant (la question (a) est deja ecrite : decommenter apres verification)
def monEtat : ERC20.State 4 := ⟨![10, 15, 20, 5], 50⟩
-- #eval monEtat.balances 0 + monEtat.balances 1 + monEtat.balances 2 + monEtat.balances 3
theorem exo3_monEtat_invariant : ERC20.supplyInvariant monEtat := by
  sorry

-- Exercice 3 : a completer
-- TODO etudiant (la question (a) est deja ecrite : decommenter apres verification)
def monEtat : ERC20.State 4 := ⟨![10, 15, 20, 5], 50⟩
-- #eval monEtat.balances 0 + monEtat.balances 1 + monEtat.balances 2 + monEtat.balances 3
theorem exo3_monEtat_invariant : ERC20.supplyInvariant monEtat := by
        ──────────────────────▶ 🟨 declaration uses `sorry`
  sorry
--% env 13
--% prove 2

Raw input:
{"cmd": "-- Exercice 3 : a completer\n-- TODO etudiant (la question (a) est deja ecrite : decommenter apres verification)\ndef monEtat : ERC20.State 4 := \u27e8![10, 15, 20, 5], 50\u27e9\n-- #eval monEtat.balances 0 + monEtat.balances 1 + monEtat.balances 2 + monEtat.balances 3\ntheorem exo3_monEtat_invariant : ERC20.supplyInvariant monEtat := by\n  sorry", "env": 12}
Raw output:
{"sorries":
 [{"proofState": 2,
   "pos": {"line": 6, "column": 2},
   "goal": "⊢ ERC20.supplyInvariant monEtat",
   "endPos": {"line": 6, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 5, "column": 8},
   "endPos": {"line": 5, "column": 30},
   "data": "declaration uses `sorry`"}],
 "env": 13}

## Annexe — structure de la preuve de l'exercice 2

*Cette annexe est séparée de l'énoncé pour que la solution ne soit pas sous les yeux avant l'effort (pattern #14161).*

**Structure de la preuve attendue** :
```lean
theorem exo2_burn_preserved : ERC20.supplyInvariant (ERC20.burn s2 0 20) := by
  have hs1 : ERC20.supplyInvariant s1 := by
    -- remontee depuis s0 via mint_preserves_supply
    sorry
  have hs2 : ERC20.supplyInvariant s2 := by
    -- remontee depuis s1 via transfer_preserves_supply (garde + distinct)
    sorry
  exact ERC20.burn_preserves_supply s2 0 20 (by decide) hs2
```

**Détail des étapes intermédiaires** :
- `hs1` : `s1 = mint s0 0 100`, `supplyInvariant s0` (état initial, exercice 1), `mint_preserves_supply s0 0 100 supplyInvariant_s0` donne `supplyInvariant s1`. Une seule application de théorème.
- `hs2` : `s2 = transfer s1 0 1 30`. Hypothèses nécessaires, **dans l'ordre où le théorème les attend** : `s1.balances 0 ≥ 30` (par unfolding de `s1`, c'est `100 ≥ 30`, trivial par `decide`), puis `(0 : Fin 3) ≠ 1` (par `decide`), et **en dernier** `supplyInvariant s1` (prouvé juste avant) — l'invariant ferme la liste, il ne l'ouvre pas.

**Pourquoi cette structure en 2 temps** : on pourrait chaîner directement `burn_preserves_supply` après une longue preuve composée, mais la structure `have` rend la trace lisible et chaque `sorry` intermédiaire correspond à une étape pédagogique claire.

**Pourquoi `by decide` pour la garde** : la garde `s2.balances 0 ≥ 20` est arithmétique : `s2.balances 0 = 70`, `70 ≥ 20` est `True` pour `decide`. Aucun lemme non-trivial nécessaire.
